# PriX (JAX) vs. Curl (PyTorch) — GPU Benchmark on Google Colab

Make sure **Runtime → Change runtime type → Hardware accelerator** is set to **T4 GPU** (or **L4 / A100 GPU**).

In [ ]:
!nvidia-smi
!pip install -q pywavelets pyyaml omegaconf onnx onnxruntime pandas scikit-learn scipy future cloudpickle
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
# Clone or update PriX and Curl, then install a JAX whose CUDA plugin matches BOTH the driver and jaxlib.
# Colab preinstalls jax + a cuda13 plugin; the Curl/CrypTen deps can replace jaxlib with a CPU build, and a
# plugin whose version differs from jaxlib's is silently ignored ("... not compatible ... will not be used"),
# which leaves JAX CPU-only. So: remove every JAX CUDA plugin, then install jax[cudaNN] for the driver's CUDA.
!if [ -d /content/prix ]; then git -C /content/prix pull; else git clone https://github.com/mugurbil/prix.git /content/prix; fi
!if [ -d /content/curl ]; then git -C /content/curl pull; else git clone https://github.com/jimouris/curl.git /content/curl; fi
!pip install -q -e /content/prix
import re, subprocess
cuda = re.search(r"CUDA Version: (\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
extra = f"cuda{cuda.group(1)}" if cuda and int(cuda.group(1)) >= 13 else "cuda12"
print("driver CUDA major:", cuda.group(1) if cuda else "?", "-> installing jax[" + extra + "]")
# Colab's preinstalled plugin is not always pip-managed, so also delete the unused plugin package directly
# (otherwise it keeps trying to register and prints "ALREADY_EXISTS: PJRT_Api already exists" in every process).
!for p in jax-cuda12-plugin jax-cuda12-pjrt jax-cuda13-plugin jax-cuda13-pjrt; do pip uninstall -q -y $p >/dev/null 2>&1; done
import glob, shutil, site, sys
unused = "xla_cuda13" if extra == "cuda12" else "xla_cuda12"
for sp in set(site.getsitepackages() + [site.getusersitepackages()]):
    for d in glob.glob(f"{sp}/jax_plugins/{unused}") + glob.glob(f"{sp}/jax_{unused.replace('xla_', '')}_p*"):
        print("removing stale plugin:", d); shutil.rmtree(d, ignore_errors=True)
!pip install -q -U "jax[{extra}]"
import importlib, jax
print("JAX", jax.__version__, "backend:", jax.default_backend(), jax.devices())
assert jax.default_backend() == "gpu", "JAX does not see the GPU: Runtime -> Restart runtime, then re-run this cell"

In [ ]:
# Run PriX GPU LLM benchmarks (Wavelet LUT mode & Fission Evaluator mode)
!python /content/prix/examples/llms/launcher.py --world_size 2 --multiprocess --model BertTiny --tensor_size 1,16 --not-full --device cuda
!python /content/prix/examples/llms/launcher.py --world_size 2 --multiprocess --model GPT2 --tensor_size 1,16 --not-full --device cuda
!python /content/prix/examples/llms/launcher.py --world_size 2 --evaluator_size 1 --multiprocess --model GPT2 --tensor_size 1,16 --not-full --device cuda

In [ ]:
# Full side-by-side GPU benchmark: PriX (JAX) vs. CrypTen (PyTorch), 5 modes, both frameworks.
# PriX uses its default peer-to-peer TCP transport; CrypTen uses gloo over TCP. ~45-60 min on a T4.
!python3 -c "import jax; assert jax.default_backend() == 'gpu', jax.devices()"
!python3 /content/prix/benchmarks/benchmark_prix_vs_crypten.py --device cuda --comm-stats --repeats 5 --llm-repeats 5 \
    --prix-root /content/prix --crypten-root /content/curl --out /content/gpu_tesla_t4.json --overwrite
from google.colab import files
files.download("/content/gpu_tesla_t4.json")  # -> copy into benchmarks/results/gpu_tesla_t4.json

In [ ]:
# Optional: GPU transport comparison (PriX over gloo and over shared memory), appended to the same file.
!python3 /content/prix/benchmarks/benchmark_prix_vs_crypten.py --device cuda --comm-stats --framework prix --prix-transport gloo --prix-root /content/prix --crypten-root /content/curl --out /content/gpu_results.json
!python3 /content/prix/benchmarks/benchmark_prix_vs_crypten.py --device cuda --comm-stats --framework prix --prix-transport shm --prix-root /content/prix --crypten-root /content/curl --out /content/gpu_results.json
!python3 /content/prix/benchmarks/transport_ablation.py /content/gpu_results.json
from google.colab import files
files.download("/content/gpu_results.json")